# 🔍 Local VDR QA Scanner - Mac M3 + VSCode

**Scans a local folder for anonymization issues**

## Setup Instructions:

### 1. Install Dependencies
```bash
pip install PyPDF2 python-docx python-pptx openpyxl pandas tqdm
```

### 2. Prepare Your Files
- Copy your VDR folder to your Mac
- Have your tracker Excel file ready

### 3. Run the notebook!
- Drag & drop files to get their paths (or type paths)
- Watch the progress bars!

In [1]:
# Install required packages (uncomment if needed)
# !pip install PyPDF2 python-docx python-pptx openpyxl pandas tqdm

In [2]:
import os
import re
import warnings
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

import PyPDF2
import docx
from pptx import Presentation

# Suppress noisy library warnings
warnings.filterwarnings("ignore")
logging.getLogger("PyPDF2").setLevel(logging.ERROR)

# App logger
logger = logging.getLogger("vdr_qa")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
    logger.addHandler(handler)

print("✅ Imports loaded!")

✅ Imports loaded!


In [3]:
# LOAD TRACKER FILE
# Drag & drop your Excel file into VSCode terminal to get the path
# Or navigate to it: e.g., /Users/yourname/Desktop/Anon_Tracker__Nautilus.xlsx

tracker_path = input("📂 Enter full path to tracker Excel file: ").strip().strip("'\"")

def load_tracker(path):
    xls = pd.ExcelFile(path)
    print(f"📑 Sheets found: {', '.join(xls.sheet_names)}")

    for name in xls.sheet_names:
        df = xls.parse(name)
        lower_cols = {c.lower(): c for c in df.columns}
        if "before" in lower_cols:
            col_name = lower_cols["before"]
            return df[col_name].dropna().unique().tolist(), name

    raise ValueError("No sheet with a 'Before' column found. Please check the file.")

if tracker_path and Path(tracker_path).exists():
    try:
        search_terms, sheet_used = load_tracker(tracker_path)
        print(f"✅ Loaded {len(search_terms)} search terms from sheet: {sheet_used}")
        print(f"\nFirst 10 terms: {search_terms[:10]}")
    except Exception as e:
        print(f"❌ Could not load tracker: {type(e).__name__}: {e}")
        search_terms = []
else:
    print("❌ File not found. Please check the path.")
    search_terms = []


📑 Sheets found: List
✅ Loaded 279 search terms from sheet: List

First 10 terms: ['(513) 900-4811', '(513) 900-5250', '(718) 921-8562', '(800) 579-1639', '(800) 690-6903', '(866) 540-7095', '(888) 776-9962', '001-35462', '26-4532998', '8500 Governor’s Hill Drive Symmes Township, Ohio 45249']


In [4]:
# PRIORITY TERMS (PREFIX PARTIAL MATCH)
# These are scanned as "starts-with" matches for concatenated tokens.
# Example: priority term "Planet" flags "PlanetFitness", but NOT "MyPlanetFitness".
# Note: this mode does NOT flag the standalone token "Planet"; include it in the tracker list for exact matches.

priority_terms = ["spirit", "airlines", "spirit aviation", "savers", "smarte", "telesis", "iae", "top gun", "aercap"
    
]

print(f"✅ Priority terms loaded: {len(priority_terms)}")


✅ Priority terms loaded: 9


In [5]:
# SELECT VDR FOLDER
# Drag & drop your VDR folder into VSCode terminal to get the path
# Or navigate to it: e.g., /Users/yourname/Desktop/Project_Vaast

vdr_folder = input("📁 Enter full path to VDR folder: ").strip().strip("'\"")

if vdr_folder and Path(vdr_folder).exists():
    print(f"✅ Selected folder: {Path(vdr_folder).name}")
    print(f"📂 Full path: {Path(vdr_folder).absolute()}")
else:
    print("❌ Folder not found. Please check the path.")
    vdr_folder = None

✅ Selected folder: Folder
📂 Full path: /Users/shahkhan/xAI/anon_qc/Folder


In [6]:
# ALL SEARCH FUNCTIONS FOR LOCAL FILES

def build_compiled_patterns(terms, mode="exact"):
    """
    Build compiled regex patterns.

    mode:
      - exact: whole-token match using non-alphanumeric boundaries
      - priority_prefix: token starts with term and has at least 1 extra token char
        Example: term 'Planet' flags 'PlanetFitness' but NOT 'MyPlanetFitness' or 'Planet'
    """
    patterns = {}
    for term in terms:
        if not isinstance(term, str):
            continue
        t = term.strip()
        if not t:
            continue

        if mode == "exact":
            pattern_str = r"(?<![0-9a-zA-Z])" + re.escape(t) + r"(?![0-9a-zA-Z])"
        elif mode == "priority_prefix":
            pattern_str = r"(?<![0-9a-zA-Z])" + re.escape(t) + r"[0-9a-zA-Z_]+"
        else:
            raise ValueError(f"Unknown mode: {mode}")

        patterns[t] = re.compile(pattern_str, flags=re.IGNORECASE)

    logger.info(f"Compiled {len(patterns)} patterns (mode={mode}).")
    return patterns


def _update_found_from_text(found, text, compiled_patterns, location_label=None, match_type=None):
    """
    Update `found` dict with matches from `text` for all compiled patterns.

    found structure:
      {
        term: {
          "count": int,
          "locations": [str, ...],
          "term_flagged": [str, ...],
          "match_type": ["exact"|"partial", ...]
        },
        ...
      }
    """
    if not text:
        return

    for term, pattern in compiled_patterns.items():
        matches = pattern.findall(text)
        if matches:
            entry = found.setdefault(term, {"count": 0, "locations": [], "term_flagged": [], "match_type": []})
            entry["count"] += len(matches)
            for m in matches:
                if m not in entry["term_flagged"]:
                    entry["term_flagged"].append(m)
            if match_type and match_type not in entry["match_type"]:
                entry["match_type"].append(match_type)
            if location_label and location_label not in entry["locations"]:
                entry["locations"].append(location_label)


def _normalize_for_match(value: str) -> str:
    """Lowercase and drop all non-alphanumerics (A-Z, a-z, 0-9)."""
    if not value:
        return ""
    return re.sub(r"[^0-9a-zA-Z]+", "", value).lower()


def _update_found_from_text_normalized(found, text, normalized_terms, location_label=None, match_type="partial"):
    """Token-level normalized scan: token_norm.startswith(term_norm)."""
    if not text or not normalized_terms:
        return

    # Tokenize from original text to keep token boundaries.
    tokens = [m.group(0) for m in re.finditer(r"[0-9A-Za-z]+", text)]
    if not tokens:
        return

    for token in tokens:
        token_norm = token.lower()
        for term, term_norm in normalized_terms.items():
            if not term_norm:
                continue
            if token_norm.startswith(term_norm):
                entry = found.setdefault(term, {"count": 0, "locations": [], "term_flagged": [], "match_type": []})
                entry["count"] += 1
                if token not in entry["term_flagged"]:
                    entry["term_flagged"].append(token)
                if match_type and match_type not in entry["match_type"]:
                    entry["match_type"].append(match_type)
                if location_label and location_label not in entry["locations"]:
                    entry["locations"].append(location_label)


def search_in_text_file(file_path, compiled_patterns, match_type=None, normalized_terms=None, chunk_size=1024 * 1024):
    """Search in .txt files using chunked reading."""
    found = {}
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                _update_found_from_text(found, chunk, compiled_patterns, location_label="CONTENT", match_type=match_type)
                _update_found_from_text_normalized(found, chunk, normalized_terms, location_label="CONTENT", match_type="partial")
    except Exception as e:
        logger.warning(f"Failed to process text file {file_path}: {type(e).__name__}: {e}")
    return found


def search_in_pdf(file_path, compiled_patterns, match_type=None, normalized_terms=None):
    """Search in PDF files page by page."""
    found = {}
    try:
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page_idx, page in enumerate(reader.pages):
                try:
                    text = page.extract_text() or ""
                except Exception as e_page:
                    logger.warning(
                        f"Failed to extract page {page_idx + 1} from {file_path}: "
                        f"{type(e_page).__name__}: {e_page}"
                    )
                    continue

                if text.strip():
                    _update_found_from_text(
                        found,
                        text,
                        compiled_patterns,
                        location_label=f"PAGE {page_idx + 1}",
                        match_type=match_type,
                    )
                    _update_found_from_text_normalized(
                        found,
                        text,
                        normalized_terms,
                        location_label=f"PAGE {page_idx + 1}",
                        match_type="partial",
                    )
    except Exception as e:
        logger.warning(f"Failed to process PDF {file_path}: {type(e).__name__}: {e}")
    return found


def search_in_word_doc(file_path, compiled_patterns, match_type=None, normalized_terms=None):
    """Search in Word documents (.docx, .doc), including headers/footers."""
    found = {}
    try:
        doc = docx.Document(file_path)

        def scan_text(text, location_label):
            if not text:
                return
            _update_found_from_text(
                found,
                text,
                compiled_patterns,
                location_label=location_label,
                match_type=match_type,
            )
            _update_found_from_text_normalized(
                found,
                text,
                normalized_terms,
                location_label=location_label,
                match_type="partial",
            )

        # Body paragraphs
        for p in doc.paragraphs:
            if p.text:
                scan_text(p.text, "BODY")

        # Body tables
        for t_idx, table in enumerate(doc.tables):
            for row in table.rows:
                for cell in row.cells:
                    if cell.text:
                        scan_text(cell.text, f"TABLE {t_idx + 1}")

        # Headers & footers (best-effort, including textboxes via XML)
        seen_part_elements = set()

        def scan_part(part, label):
            elem = getattr(part, "_element", None)
            if elem is None:
                return
            key = id(elem)
            if key in seen_part_elements:
                return
            seen_part_elements.add(key)

            texts = []
            for el in elem.iter():
                if getattr(el, "text", None) and str(getattr(el, "tag", "")).endswith("}t"):
                    texts.append(el.text)
            scan_text(" ".join(texts), label)

        for s_idx, section in enumerate(doc.sections, 1):
            scan_part(section.header, f"HEADER {s_idx}")
            scan_part(section.footer, f"FOOTER {s_idx}")
            scan_part(section.first_page_header, f"FIRST_PAGE_HEADER {s_idx}")
            scan_part(section.first_page_footer, f"FIRST_PAGE_FOOTER {s_idx}")
            scan_part(section.even_page_header, f"EVEN_PAGE_HEADER {s_idx}")
            scan_part(section.even_page_footer, f"EVEN_PAGE_FOOTER {s_idx}")
    except Exception as e:
        logger.warning(f"Failed to process Word doc {file_path}: {type(e).__name__}: {e}")
    return found


def search_in_excel(file_path, compiled_patterns, match_type=None, normalized_terms=None):
    """Search in Excel files (.xlsx, .xls) using only string-like columns."""
    found = {}
    try:
        df_dict = pd.read_excel(file_path, sheet_name=None)
        for sheet_name, df in df_dict.items():
            # Only string/object columns
            obj_cols = df.select_dtypes(include=["object"]).columns
            if not len(obj_cols):
                continue

            # Flatten object columns into a single string
            try:
                sub = df[obj_cols].astype(str).fillna("")
                sheet_text = " ".join(sub.values.ravel().tolist())
            except Exception as e_sheet:
                logger.warning(
                    f"Failed to build text for sheet {sheet_name} "
                    f"in {file_path}: {type(e_sheet).__name__}: {e_sheet}"
                )
                continue

            if not sheet_text.strip():
                continue

            _update_found_from_text(found, sheet_text, compiled_patterns, location_label=sheet_name, match_type=match_type)
            _update_found_from_text_normalized(found, sheet_text, normalized_terms, location_label=sheet_name, match_type="partial")
    except Exception as e:
        logger.warning(f"Failed to process Excel file {file_path}: {type(e).__name__}: {e}")
    return found


def search_in_csv(file_path, compiled_patterns, match_type=None, normalized_terms=None):
    """Search in CSV files via pandas."""
    found = {}
    try:
        df = pd.read_csv(file_path, dtype=str, encoding_errors="ignore")
        text = " ".join(df.astype(str).fillna("").values.ravel().tolist())
        _update_found_from_text(found, text, compiled_patterns, location_label="CONTENT", match_type=match_type)
        _update_found_from_text_normalized(found, text, normalized_terms, location_label="CONTENT", match_type="partial")
    except Exception as e:
        logger.warning(f"Failed to process CSV file {file_path}: {type(e).__name__}: {e}")
    return found


def search_in_powerpoint(file_path, compiled_patterns, match_type=None, normalized_terms=None):
    """Search in PowerPoint files (.pptx, .ppt), including layout/master and notes."""
    found = {}
    try:
        prs = Presentation(file_path)

        def scan_text(text, location_label):
            if not text:
                return
            _update_found_from_text(
                found,
                text,
                compiled_patterns,
                location_label=location_label,
                match_type=match_type,
            )
            _update_found_from_text_normalized(
                found,
                text,
                normalized_terms,
                location_label=location_label,
                match_type="partial",
            )

        def scan_shapes(shapes, location_label):
            for shape in shapes:
                try:
                    if hasattr(shape, "text") and shape.text:
                        scan_text(shape.text, location_label)

                    if hasattr(shape, "has_table") and shape.has_table:
                        table = shape.table
                        for row in table.rows:
                            for cell in row.cells:
                                if cell.text:
                                    scan_text(cell.text, f"{location_label} TABLE")
                except Exception as e_shape:
                    logger.debug(
                        f"Shape parse issue in {file_path} at {location_label}: "
                        f"{type(e_shape).__name__}: {e_shape}"
                    )

        scanned_layouts = set()
        scanned_masters = set()

        for slide_idx, slide in enumerate(prs.slides):
            slide_label = f"SLIDE {slide_idx + 1}"

            # Slide shapes
            scan_shapes(slide.shapes, slide_label)

            # Speaker notes
            try:
                if getattr(slide, "has_notes_slide", False) and slide.notes_slide:
                    scan_shapes(slide.notes_slide.shapes, f"{slide_label} NOTES")
            except Exception:
                pass

            # Layout & master shapes: dedupe across slides for speed
            try:
                layout = slide.slide_layout
                layout_key = str(layout.part.partname)
                if layout_key not in scanned_layouts:
                    scanned_layouts.add(layout_key)
                    scan_shapes(layout.shapes, f"LAYOUT {layout_key}")
            except Exception:
                pass
            try:
                master = slide.slide_master
                master_key = str(master.part.partname)
                if master_key not in scanned_masters:
                    scanned_masters.add(master_key)
                    scan_shapes(master.shapes, f"MASTER {master_key}")
            except Exception:
                pass
    except Exception as e:
        logger.warning(
            f"Failed to process PowerPoint file {file_path}: {type(e).__name__}: {e}"
        )
    return found


def get_search_function(file_path):
    """Route to correct search function based on file extension."""
    ext = Path(file_path).suffix.lower()

    # Legacy format warning
    if ext in [".doc", ".xls", ".ppt"]:
        logger.warning(
            f"Legacy format {ext} may not be fully supported for {file_path}. "
            f"Convert to a modern format (.docx/.xlsx/.pptx) for best results."
        )

    mapping = {
        ".pdf": search_in_pdf,
        ".docx": search_in_word_doc,
        ".doc": search_in_word_doc,
        ".xlsx": search_in_excel,
        ".xls": search_in_excel,
        ".pptx": search_in_powerpoint,
        ".ppt": search_in_powerpoint,
        ".csv": search_in_csv,
        ".txt": search_in_text_file,
    }

    return mapping.get(ext, None)


def scan_vdr_folder(vdr_path, search_terms, priority_terms=None, exclude_folders=None, use_parallel=True, max_workers=None):
    """
    Recursively scan VDR folder for search terms.

    - Precompiles regex patterns once.
    - Scans filenames and file content.
    - Uses thread pool for parallelism by default (safe in notebooks).
    """
    vdr_path = Path(vdr_path)

    if exclude_folders is None:
        exclude_folders = {".git", "__pycache__", "node_modules", ".venv", "venv"}

    exact_patterns = build_compiled_patterns(search_terms, mode="exact")
    priority_patterns = build_compiled_patterns(priority_terms or [], mode="priority_prefix") if priority_terms else {}
    normalized_terms = {t: _normalize_for_match(t) for t in (search_terms or []) if isinstance(t, str) and re.search(r"[^0-9A-Za-z]", t) and _normalize_for_match(t)}
    normalized_priority_terms = {t: _normalize_for_match(t) for t in (priority_terms or []) if isinstance(t, str) and re.search(r"[^0-9A-Za-z]", t) and _normalize_for_match(t)}

    if not exact_patterns and not priority_patterns:
        logger.warning("No valid patterns compiled (exact or priority).")
        return [], 0

    # Discover files
    all_files = []
    for root, dirs, files in os.walk(vdr_path):
        dirs[:] = [d for d in dirs if d not in exclude_folders]
        for name in files:
            file_path = Path(root) / name
            if get_search_function(file_path) is not None:
                all_files.append(file_path)

    logger.info(f"Discovered {len(all_files)} files to scan.")

    def process_file(file_path: Path):
        found_terms = {}

        # 1) Filename scan (exact + priority prefix)
        if exact_patterns:
            _update_found_from_text(found_terms, file_path.name, exact_patterns, location_label="FILENAME", match_type="exact")
        if priority_patterns:
            _update_found_from_text(found_terms, file_path.name, priority_patterns, location_label="FILENAME", match_type="partial")
        if normalized_terms:
            _update_found_from_text_normalized(found_terms, file_path.name, normalized_terms, location_label="FILENAME", match_type="partial")
        if normalized_priority_terms:
            _update_found_from_text_normalized(found_terms, file_path.name, normalized_priority_terms, location_label="FILENAME", match_type="partial")

        # 2) Content scan (exact + priority prefix)
        search_fn = get_search_function(file_path)
        if search_fn is not None:
            content_found = {}
            for patterns, match_type, norm_terms in ((exact_patterns, "exact", normalized_terms), (priority_patterns, "partial", normalized_priority_terms)):
                if not patterns:
                    continue
                part_found = search_fn(file_path, patterns, match_type=match_type, normalized_terms=norm_terms)
                for term, info in part_found.items():
                    merged = content_found.setdefault(term, {"count": 0, "locations": [], "term_flagged": [], "match_type": []})
                    merged["count"] += info.get("count", 0)
                    for loc in info.get("locations", []):
                        if loc not in merged["locations"]:
                            merged["locations"].append(loc)
                    for flagged in info.get("term_flagged", []):
                        if flagged not in merged["term_flagged"]:
                            merged["term_flagged"].append(flagged)
                    for mt in info.get("match_type", []):
                        if mt not in merged["match_type"]:
                            merged["match_type"].append(mt)

            for term, info in content_found.items():
                entry = found_terms.setdefault(term, {"count": 0, "locations": [], "term_flagged": [], "match_type": []})
                entry["count"] += info.get("count", 0)
                for loc in info.get("locations", []):
                    if loc not in entry["locations"]:
                        entry["locations"].append(loc)
                for flagged in info.get("term_flagged", []):
                    if flagged not in entry["term_flagged"]:
                        entry["term_flagged"].append(flagged)
                for mt in info.get("match_type", []):
                    if mt not in entry["match_type"]:
                        entry["match_type"].append(mt)

        if not found_terms:
            return None

        try:
            rel_path = file_path.relative_to(vdr_path)
        except ValueError:
            rel_path = file_path

        return {
            "file_name": file_path.name,
            "file_path": str(file_path.absolute()),
            "relative_path": str(rel_path),
            "file_type": file_path.suffix[1:].lower(),  # strip dot
            "found_terms": found_terms,
        }

    results = []

    if use_parallel and all_files:
        max_workers = max_workers or min(8, os.cpu_count() or 4)
        logger.info(f"Scanning with ThreadPoolExecutor, max_workers={max_workers} ...")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_file, fp): fp for fp in all_files}
            for future in tqdm(as_completed(futures), total=len(futures), desc="Scanning files"):
                res = future.result()
                if res:
                    results.append(res)
    else:
        logger.info("Scanning sequentially ...")
        for fp in tqdm(all_files, desc="Scanning files"):
            res = process_file(fp)
            if res:
                results.append(res)

    return results, len(all_files)


print("✅ All functions loaded!")

✅ All functions loaded!


In [7]:
# RUN THE SCAN

if not vdr_folder:
    print("❌ No VDR folder selected. Please run the folder selection cell above.")
elif not search_terms:
    print("❌ No search terms loaded. Please run the tracker cell above.")
else:
    print("\n" + "=" * 60)
    print("🚀 STARTING QA SEARCH")
    print("=" * 60)
    print(f"VDR Folder: {Path(vdr_folder).name}")
    print(f"Search terms: {len(search_terms)}")
    priority_terms = priority_terms if "priority_terms" in locals() else []
    print(f"Priority prefix terms: {len(priority_terms)}")
    print("=" * 60)

    results, total_files = scan_vdr_folder(
        vdr_folder,
        search_terms,
        priority_terms=priority_terms,
        use_parallel=True,   # set to False if you want purely sequential
    )

    print("\n" + "=" * 60)
    print("✅ SCAN COMPLETE")
    print("=" * 60)
    print(f"Files scanned: {total_files}")
    print(f"Files with issues: {len(results)}")
    print("=" * 60)

INFO - Compiled 279 patterns (mode=exact).
INFO - Compiled 9 patterns (mode=priority_prefix).
INFO - Discovered 1 files to scan.
INFO - Scanning with ThreadPoolExecutor, max_workers=8 ...



🚀 STARTING QA SEARCH
VDR Folder: Folder
Search terms: 279
Priority prefix terms: 9


Scanning files:   0%|          | 0/1 [00:00<?, ?it/s]


✅ SCAN COMPLETE
Files scanned: 1
Files with issues: 1


In [8]:
# DISPLAY & EXPORT RESULTS

if "results" not in locals():
    print("⚠️ Please run the scan cell above first.")
elif not results:
    print("\n✅ NO ISSUES FOUND! All documents properly anonymized.")
else:
    print(f"\n⚠️ FOUND {len(results)} FILES WITH POTENTIAL ISSUES:\n")

    # Pretty-print to console
    for idx, result in enumerate(results, 1):
        print("\n" + "=" * 60)
        print(f"[{idx}] {result['file_name']}")
        print("=" * 60)
        print(f"Path: {result['relative_path']}")
        print(f"Type: {result['file_type']}")
        print("Issues:")
        for term, info in result["found_terms"].items():
            count = info.get("count", 0)
            locations = info.get("locations", [])
            print(f"  • '{term}': {count} occurrence(s)")
            if locations:
                print(f"      Locations: {', '.join(locations)}")

    # Flatten for export
    export_rows = []
    for result in results:
        for term, info in result["found_terms"].items():
            export_rows.append(
                {
                    "file_name": result["file_name"],
                    "relative_path": result["relative_path"],
                    "file_type": result["file_type"],
                    "anon_term": term,
                    "term_flagged": "; ".join(info.get("term_flagged", [])),
                    "match_type": "partial" if "partial" in info.get("match_type", []) else "exact",
                    "count": info.get("count", 0),
                    "locations": "; ".join(info.get("locations", [])),
                }
            )

    df_export = pd.DataFrame(export_rows)
    output_file = "vdr_qa_results.csv"
    df_export.to_csv(output_file, index=False)

    print("\n" + "=" * 60)
    print(f"💾 Exported {len(export_rows)} term hits to: {output_file}")
    print(f"📂 Location: {Path(output_file).absolute()}")
    print("=" * 60)

    total_occurrences = sum(row["count"] for row in export_rows)
    unique_terms = sorted({row["anon_term"] for row in export_rows})

    print("\n📊 Summary")
    print(f"   Files scanned: {total_files if 'total_files' in locals() else 'N/A'}")
    print(f"   Files with issues: {len(results)}")
    print(f"   Total term occurrences: {total_occurrences}")
    print(f"   Unique terms found: {len(unique_terms)}")


⚠️ FOUND 1 FILES WITH POTENTIAL ISSUES:


[1] March 31, 2017 Form 10-Q.docx
Path: March 31, 2017 Form 10-Q.docx
Type: docx
Issues:
  • 'Vantiv': 8 occurrence(s)
      Locations: HEADER 11, HEADER 18, HEADER 23, HEADER 30

💾 Exported 1 term hits to: vdr_qa_results.csv
📂 Location: /Users/shahkhan/xAI/anon_qc/vdr_qa_results.csv

📊 Summary
   Files scanned: 1
   Files with issues: 1
   Total term occurrences: 8
   Unique terms found: 1


In [9]:
# OPTIONAL: VIEW SPECIFIC FILE DETAILS

def view_file_details(file_number):
    """View detailed results for a specific file"""
    if 'results' not in locals() or not results:
        print("No results available")
        return
    
    if file_number < 1 or file_number > len(results):
        print(f"Invalid file number. Choose between 1 and {len(results)}")
        return
    
    result = results[file_number - 1]
    
    print("\n" + "="*60)
    print(f"FILE DETAILS: {result['file_name']}")
    print("="*60)
    print(f"Path: {result['relative_path']}")
    print(f"Full Path: {result['file_path']}")
    print(f"Type: {result['file_type']}")
    print(f"\nFound Terms:")
    
    for term, info in result["found_terms"].items():
        count = info.get("count", 0)
        print(f"\n  '{term}': {count} occurrence(s)")
        flagged = info.get("term_flagged", [])
        if flagged:
            print(f"      Flagged: {', '.join(flagged)}")
        locations = info.get("locations", [])
        if locations:
            print(f"      Locations: {', '.join(locations)}")
    
    print("\n" + "="*60)

print("💡 Usage: view_file_details(1) to see details of first file")

💡 Usage: view_file_details(1) to see details of first file
